In [2]:
import pandas as pd
import json
from pathlib import Path

# Resolver caminhos do projeto
BASE_DIR = Path.cwd()
if BASE_DIR.name == "Extraction":
    BASE_DIR = BASE_DIR.parents[1]
elif BASE_DIR.name == "scripts":
    BASE_DIR = BASE_DIR.parent

config_path = BASE_DIR / "Config" / "sheets-id.json"
with open(config_path, encoding="utf-8") as f:
    data = json.load(f)

lst_months = ["Janeiro","Fevereiro","Março","Abril","Maio","Junho","Julho","Agosto","Setembro","Outubro","Novembro","Dezembro"]

despesas_df = pd.DataFrame()

for i in data["Sheets"]:
    id = data["Sheets"][i]
    url = f"https://docs.google.com/spreadsheets/d/{id}/export?format=xlsx"
    print(f"Importando dados da planilha: {i} - ID: {id}")

    for month in lst_months:
        print(f"Importando dados do mês: {month}")
        temp_df = pd.DataFrame()
        temp_df = pd.read_excel(url, sheet_name=month, header=1)
        temp_df = temp_df.drop(temp_df.columns[5:], axis=1)
        temp_df = temp_df.dropna(how="all")
        despesas_df = pd.concat([despesas_df, temp_df], ignore_index=True)


Importando dados da planilha: 2026 - ID: 1v6t5Q5pqdftjTzz4SK1EP4w1QorYSu8g1MdDinRciLk
Importando dados do mês: Janeiro
Importando dados do mês: Fevereiro
Importando dados do mês: Março
Importando dados do mês: Abril
Importando dados do mês: Maio
Importando dados do mês: Junho
Importando dados do mês: Julho
Importando dados do mês: Agosto
Importando dados do mês: Setembro
Importando dados do mês: Outubro
Importando dados do mês: Novembro
Importando dados do mês: Dezembro


C:\Users\Lucas\AppData\Local\Temp\ipykernel_24548\3143519256.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  despesas_df = pd.concat([despesas_df, temp_df], ignore_index=True)


Importando dados da planilha: 2025 - ID: 1QyEE01i2u2YhJ-nOdhWkOSflNRqQKHPB5vzyMRayqMg
Importando dados do mês: Janeiro
Importando dados do mês: Fevereiro
Importando dados do mês: Março
Importando dados do mês: Abril
Importando dados do mês: Maio
Importando dados do mês: Junho
Importando dados do mês: Julho
Importando dados do mês: Agosto
Importando dados do mês: Setembro
Importando dados do mês: Outubro
Importando dados do mês: Novembro
Importando dados do mês: Dezembro
Importando dados da planilha: 2024 - ID: 1pzkumLWs0_MQB9bk-JpF0TInWG8jI_5FU7HEXJhGeHI
Importando dados do mês: Janeiro
Importando dados do mês: Fevereiro
Importando dados do mês: Março
Importando dados do mês: Abril
Importando dados do mês: Maio
Importando dados do mês: Junho
Importando dados do mês: Julho
Importando dados do mês: Agosto
Importando dados do mês: Setembro
Importando dados do mês: Outubro
Importando dados do mês: Novembro
Importando dados do mês: Dezembro


In [ ]:
# Ordenar por data e salvar backup em CSV
df_sort = despesas_df.sort_values(by=['Data'], ascending=True)

csv_path = BASE_DIR / "database" / "historical-despesas.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)
df_sort.to_csv(csv_path, index=False, sep=";", encoding="utf-8")
print(f"Backup CSV salvo com sucesso em: {csv_path}")


In [ ]:
import sys
import sqlite3

# Adicionar caminho do backend para importar dbconnect
backend_dir = str(BASE_DIR / "scripts" / "backend")
if backend_dir not in sys.path:
    sys.path.append(backend_dir)

import dbconnect as db

# Se df_sort não estiver em memória, lê do arquivo CSV
if 'df_sort' not in globals() or df_sort.empty:
    csv_path = BASE_DIR / "database" / "historical-despesas.csv"
    df_sort = pd.read_csv(csv_path, sep=";", encoding="utf-8")

# 1. Conectar ao SQLite e garantir criação das tabelas
conn = db.connect_db()
cursor = conn.cursor()
db.create_tables(cursor)

# 2. Limpeza e normalização das colunas
df_clean = df_sort.copy()
# Garantir nomes padrão de colunas independente de acentuação na leitura
df_clean.columns = ['Data', 'Valor', 'Pagamento', 'Descricao', 'Categoria']
df_clean = df_clean.dropna(subset=['Data', 'Valor']).copy()

df_clean['Data'] = pd.to_datetime(df_clean['Data']).dt.strftime('%Y-%m-%d')
df_clean['Valor'] = pd.to_numeric(df_clean['Valor'], errors='coerce')
df_clean['Pagamento'] = df_clean['Pagamento'].astype(str).str.strip()
df_clean['Descricao'] = df_clean['Descricao'].fillna('').astype(str).str.strip()
df_clean['Categoria'] = df_clean['Categoria'].astype(str).str.strip()

# Mapear e padronizar categorias com nomes limpos
cat_norm_map = {
    'lazer/vontades': 'Lazer/Vontades',
    'necessidades': 'Necessidades',
    'investimento': 'Investimentos',
    'investimentos': 'Investimentos'
}
df_clean['Categoria_Norm'] = df_clean['Categoria'].str.lower().map(cat_norm_map).fillna(df_clean['Categoria'])

# 3. Registrar Formas de Pagamento e Categorias
for forma in df_clean['Pagamento'].unique():
    db.get_or_create_forma_pagamento(cursor, forma)

for cat in df_clean['Categoria_Norm'].unique():
    db.get_or_create_categoria(cursor, cat)

conn.commit()

# 4. Mapear chaves estrangeiras (IDs)
cursor.execute("SELECT nome, id FROM forma_pagamento")
pagamento_map = dict(cursor.fetchall())

cursor.execute("SELECT nome, id FROM categoria")
categoria_map = dict(cursor.fetchall())

df_clean['forma_pagamento_id'] = df_clean['Pagamento'].map(pagamento_map)
df_clean['categoria_id'] = df_clean['Categoria_Norm'].map(categoria_map)

# 5. Inserir despesas (limpa dados anteriores para permitir reprocessamento seguro)
cursor.execute("DELETE FROM despesas")
cursor.execute("DELETE FROM sqlite_sequence WHERE name='despesas'")

records = [
    (row['Data'], float(row['Valor']), str(row['Descricao']), int(row['forma_pagamento_id']), int(row['categoria_id']))
    for _, row in df_clean.iterrows()
]

cursor.executemany(
    "INSERT INTO despesas (data, valor, descricao, forma_pagamento_id, categoria_id) VALUES (?, ?, ?, ?, ?)",
    records
)
conn.commit()

print(f"Carga concluída no banco de dados!")
print(f"Total de registros inseridos: {len(records)}")
print(f"Categorias cadastradas: {categoria_map}")
print(f"Formas de pagamento cadastradas: {pagamento_map}")


In [3]:
# Validação dos dados importados: Resumo por Categoria
resumo_df = pd.read_sql_query("""
    SELECT 
        c.nome AS Categoria,
        COUNT(d.id) AS Total_Lancamentos,
        ROUND(SUM(d.valor), 2) AS Valor_Total
    FROM despesas d
    JOIN categoria c ON d.categoria_id = c.id
    GROUP BY c.nome
    ORDER BY Valor_Total DESC
""", conn)

print("=== Resumo por Categoria ===")
print(resumo_df.to_string(index=False))

# Amostra das últimas despesas
preview_df = pd.read_sql_query("""
    SELECT d.id, d.data, d.valor, d.descricao, f.nome AS forma_pagamento, c.nome AS categoria
    FROM despesas d
    JOIN forma_pagamento f ON d.forma_pagamento_id = f.id
    JOIN categoria c ON d.categoria_id = c.id
    ORDER BY d.data DESC
    LIMIT 5
""", conn)

print("\n=== Amostra das Últimas 5 Despesas ===")
print(preview_df.to_string(index=False))


NameError: name 'conn' is not defined